# Severe YouTube Affinity Index vs CamiHawke

This notebook builds comparable severe-taxonomy macro-topic vectors for each YouTube channel and computes cosine similarity against CamiHawke.

## 1. Setup

This section imports the libraries and defines the working paths used throughout the analysis.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
CAMI_FILE = BASE_DIR / "CamiHawke_macro_topics_comparable_v3.xlsx"
OUTPUT_CSV = BASE_DIR / "youtube_affinity_index_vs_camihawke.csv"
VALID_ROLES = {"Core", "Secondary", "Niche"}

channel_files = sorted(
    path
    for path in BASE_DIR.glob("*.xlsx")
    if path.name != CAMI_FILE.name
)

print(f"Base directory: {BASE_DIR}")
print(f"Channel files found: {len(channel_files)}")

Base directory: c:\Users\miche\Desktop\A4B LAB\Project A4B\Affinity Index\Severe affinity index
Channel files found: 25


## 2. Helper Functions

This section defines reusable functions to read the Excel files, standardize column detection, normalize topic weights, and compute cosine similarity.

In [2]:
def normalize_label(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    normalized_columns = {normalize_label(column): column for column in df.columns}
    for candidate in candidates:
        key = normalize_label(candidate)
        if key in normalized_columns:
            return normalized_columns[key]
    raise KeyError(f"None of the candidate columns were found: {candidates}")


def load_table_with_detected_header(path: Path, sheet_name: str, required_columns: list[str]) -> pd.DataFrame:
    for header_row in (0, 1):
        df = pd.read_excel(path, sheet_name=sheet_name, header=header_row)
        normalized_columns = {normalize_label(column) for column in df.columns}
        if all(normalize_label(column) in normalized_columns for column in required_columns):
            return df
    raise ValueError(f"Could not detect a valid header for {path.name} / {sheet_name}")


def extract_channel_name(path: Path) -> str:
    return re.sub(r"_macro_topics?_v\d+$", "", path.stem, flags=re.IGNORECASE)


def read_channel_macro_topics(path: Path) -> pd.DataFrame:
    df = load_table_with_detected_header(
        path,
        sheet_name="Macro Topics",
        required_columns=["Macro Topic", "Role"],
    )
    macro_col = find_column(df, ["Macro-topic", "Macro Topic"])
    weight_col = find_column(df, ["Video Coverage %", "Post Coverage %", "Videos Covered", "Unique Videos Covered"])

    cleaned = df[[macro_col, weight_col]].copy()
    cleaned.columns = ["macro_topic", "raw_weight"]
    cleaned["macro_topic"] = cleaned["macro_topic"].astype(str).str.strip()
    cleaned["raw_weight"] = pd.to_numeric(cleaned["raw_weight"], errors="coerce").fillna(0.0)
    cleaned = cleaned[cleaned["macro_topic"].ne("")]
    return cleaned


def read_camihawke_youtube(path: Path) -> pd.DataFrame:
    df = load_table_with_detected_header(
        path,
        sheet_name="YouTube",
        required_columns=["Macro Topic", "Role"],
    )
    macro_col = find_column(df, ["Macro Topic", "Macro-topic"])
    role_col = find_column(df, ["Role"])
    weight_col = find_column(df, ["Video Coverage %", "Post Coverage %", "Unique Videos Covered", "Videos Covered"])

    cleaned = df[df[role_col].astype(str).str.strip().isin(VALID_ROLES)][[macro_col, weight_col]].copy()
    cleaned.columns = ["macro_topic", "raw_weight"]
    cleaned["macro_topic"] = cleaned["macro_topic"].astype(str).str.strip()
    cleaned["raw_weight"] = pd.to_numeric(cleaned["raw_weight"], errors="coerce").fillna(0.0)
    cleaned = cleaned[cleaned["macro_topic"].ne("")]
    return cleaned


def build_normalized_vector(df: pd.DataFrame, topic_universe: list[str]) -> pd.Series:
    aggregated = df.groupby("macro_topic", as_index=True)["raw_weight"].sum()
    vector = pd.Series(0.0, index=topic_universe, dtype=float)
    shared_topics = aggregated.index.intersection(vector.index)
    vector.loc[shared_topics] = aggregated.loc[shared_topics].astype(float)

    total_weight = vector.sum()
    if total_weight == 0:
        raise ValueError("The vector has zero total weight and cannot be normalized.")

    return vector / total_weight


def cosine_similarity(left: pd.Series, right: pd.Series) -> float:
    numerator = float(np.dot(left.values, right.values))
    denominator = float(np.linalg.norm(left.values) * np.linalg.norm(right.values))
    if denominator == 0:
        return np.nan
    return numerator / denominator

## 3. Define the Comparable Topic Universe

This section collects the final macro-topics used across the Severe YouTube channel files and checks that the comparable universe contains exactly 27 topics.

In [3]:
topic_universe = sorted(
    {
        topic
        for path in channel_files
        for topic in read_channel_macro_topics(path)["macro_topic"].tolist()
    }
)

assert len(topic_universe) == 27, f"Expected 27 comparable macro-topics, found {len(topic_universe)}"

topic_universe

['Animals & Nature',
 'Beauty & Fashion',
 'Challenges',
 'Creator Life',
 'Daily Vlogs',
 'Digital Culture',
 'Family Dynamics',
 'Food & Cooking',
 'Gaming',
 'Health & Wellbeing',
 'Home',
 'Knowledge & Critical Thinking',
 'Mental Health',
 'Music',
 'Parody',
 'Personal Growth',
 'Podcast',
 'Pop Culture & Media',
 'Reaction',
 'Relatable Comedy',
 'Relationships',
 'Shopping',
 'Social Issues',
 'Sport',
 'Storytime',
 'Travel',
 'Work & Education']

## 4. Build the CamiHawke Reference Vector

This section reads only the `YouTube` sheet for CamiHawke, keeps the valid macro-topic summary rows, and builds the normalized 27-topic vector.

In [4]:
camihawke_topics = read_camihawke_youtube(CAMI_FILE)
camihawke_vector = build_normalized_vector(camihawke_topics, topic_universe)

camihawke_vector.to_frame(name="CamiHawke_weight").T

,Animals & Nature,Beauty & Fashion,Challenges,Creator Life,Daily Vlogs,Digital Culture,Family Dynamics,Food & Cooking,Gaming,Health & Wellbeing,...,Pop Culture & Media,Reaction,Relatable Comedy,Relationships,Shopping,Social Issues,Sport,Storytime,Travel,Work & Education
CamiHawke_weight,0.0,0.0,0.0,0.159091,0.0,0.022727,0.0,0.022727,0.113636,0.0,...,0.227273,0.0,0.227273,0.068182,0.0,0.0,0.022727,0.022727,0.022727,0.0


## 5. Build Channel Vectors and Compute Cosine Similarity

This section creates one normalized 27-topic vector per channel and computes the affinity index as cosine similarity versus CamiHawke.

In [5]:
results = []
channel_vectors = {}

for path in channel_files:
    channel_name = extract_channel_name(path)
    channel_topics = read_channel_macro_topics(path)
    channel_vector = build_normalized_vector(channel_topics, topic_universe)
    affinity_index = cosine_similarity(camihawke_vector, channel_vector)

    channel_vectors[channel_name] = channel_vector
    results.append(
        {
            "channel_name": channel_name,
            "affinity_index": affinity_index,
        }
    )

results_df = pd.DataFrame(results).sort_values("affinity_index", ascending=False).reset_index(drop=True)
results_df

,channel_name,affinity_index
0,MrMarra,0.595571
1,IlMatricomio,0.593031
2,AdrianoMoretti,0.587484
3,RaissaMomo,0.583374
4,SaraLaRusca,0.581467
5,THOMASBASILICO,0.466809
6,roccotnl,0.428629
7,ContenutiZero,0.305267
8,ValentinaBarbieri,0.298230
9,CanaleDiVenti,0.285185


## 6. Export the Ranked Affinity Index CSV

This section saves the affinity index table in descending order so it can be reused outside the notebook.

In [6]:
results_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV to: {OUTPUT_CSV}")
results_df.head(10)

Saved CSV to: c:\Users\miche\Desktop\A4B LAB\Project A4B\Affinity Index\Severe affinity index\youtube_affinity_index_vs_camihawke.csv


,channel_name,affinity_index
0,MrMarra,0.595571
1,IlMatricomio,0.593031
2,AdrianoMoretti,0.587484
3,RaissaMomo,0.583374
4,SaraLaRusca,0.581467
5,THOMASBASILICO,0.466809
6,roccotnl,0.428629
7,ContenutiZero,0.305267
8,ValentinaBarbieri,0.298230
9,CanaleDiVenti,0.285185
